In [1]:
from pathlib import Path
from collections import Counter

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.infer_shapes import InferShapes

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
INT_DIR = BUILD_DIR / "intermediate_models"

CANDIDATES = [
    INT_DIR / "step_tidy_up.onnx",
    INT_DIR / "step_tidy_up_no_fold.onnx",
    INT_DIR / "step_streamline.onnx",
    INT_DIR / "step_convert_to_hw.onnx",
]

def print_ops(path):
    model = ModelWrapper(str(path))
    ops = Counter(n.op_type for n in model.graph.node)
    print(f"\n=== OPS: {path.name} ===")
    for k, v in sorted(ops.items()):
        print(f"{k}: {v}")

def print_first_matching_nodes(path, op_types=None, max_print=20):
    model = ModelWrapper(str(path))
    print(f"\n=== MATCHING NODES: {path.name} ===")
    cnt = 0
    for i, n in enumerate(model.graph.node):
        if op_types is None or n.op_type in op_types:
            print(f"\n[{i}] {n.name if n.name else '<unnamed>'}")
            print("  op_type:", n.op_type)
            print("  domain :", n.domain if hasattr(n, "domain") else "")
            print("  inputs :", list(n.input))
            print("  outputs:", list(n.output))
            cnt += 1
            if cnt >= max_print:
                break
    if cnt == 0:
        print("No matching nodes.")

def tensor_info(model, tname):
    print(f"\n[tensor] {tname}")
    try:
        print("  dtype :", model.get_tensor_datatype(tname))
    except Exception as e:
        print("  dtype :", e)
    try:
        print("  shape :", model.get_tensor_shape(tname))
    except Exception as e:
        print("  shape :", e)
    try:
        prod = model.find_producer(tname)
        if prod is None:
            print("  producer: None")
        else:
            print("  producer:", prod.name if prod.name else "<unnamed>", "|", prod.op_type, "|", prod.domain if hasattr(prod, "domain") else "")
    except Exception as e:
        print("  producer:", e)
    try:
        cons = model.find_consumers(tname)
        if cons is None or len(cons) == 0:
            print("  consumers: []")
        else:
            for c in cons:
                print("  consumer:", c.name if c.name else "<unnamed>", "|", c.op_type, "|", c.domain if hasattr(c, "domain") else "")
    except Exception as e:
        print("  consumers:", e)

def find_tensor_names(path, keywords):
    model = ModelWrapper(str(path))
    all_names = set()

    for vi in list(model.graph.input) + list(model.graph.output) + list(model.graph.value_info):
        all_names.add(vi.name)
    for init in model.graph.initializer:
        all_names.add(init.name)
    for n in model.graph.node:
        for x in n.input:
            all_names.add(x)
        for x in n.output:
            all_names.add(x)

    print(f"\n=== TENSOR SEARCH: {path.name} ===")
    for name in sorted(all_names):
        low = name.lower()
        if any(k.lower() in low for k in keywords):
            print(name)

def inspect_file(path):
    print("\n\n##################################################")
    print("FILE:", path)
    print("##################################################")

    model = ModelWrapper(str(path))

    # 再跑一次推断，避免有些中间文件 datatype 没刷新
    try:
        model = model.transform(InferShapes())
    except Exception as e:
        print("[WARN] InferShapes:", e)
    try:
        model = model.transform(InferDataTypes())
    except Exception as e:
        print("[WARN] InferDataTypes:", e)

    print_ops(path)
    print_first_matching_nodes(path, op_types=["MultiThreshold", "Im2Col", "MatMul", "Conv", "Transpose", "Pad"], max_print=40)
    find_tensor_names(path, ["multithreshold", "im2col", "matmul", "unsqueeze", "global_in"])

    # 重点盯这些名字
    likely_tensors = [
        "global_in",
        "Unsqueeze_0_out0",
        "MultiThreshold_0_out0",
        "Im2Col_0_out0",
        "MatMul_1_out0",
    ]

    for t in likely_tensors:
        tensor_info(model, t)

    # 再自动检查所有 MultiThreshold / Im2Col / MatMul 的输入输出
    print(f"\n=== AUTO TRACE IMPORTANT TENSORS: {path.name} ===")
    for n in model.graph.node:
        if n.op_type in ["MultiThreshold", "Im2Col", "MatMul"]:
            print(f"\nNODE: {n.name if n.name else '<unnamed>'} | {n.op_type}")
            for t in list(n.input) + list(n.output):
                tensor_info(model, t)

def main():
    for p in CANDIDATES:
        if p.exists():
            inspect_file(p)
        else:
            print(f"[SKIP] not found: {p}")

if __name__ == "__main__":
    main()



##################################################
FILE: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_tidy_up.onnx
##################################################

=== OPS: step_tidy_up.onnx ===
Add: 4
Conv: 9
Div: 10
Flatten: 1
MatMul: 1
Mul: 27
MultiThreshold: 13
Pad: 6
Unsqueeze: 1

=== MATCHING NODES: step_tidy_up.onnx ===

[1] MultiThreshold_0
  op_type: MultiThreshold
  domain : qonnx.custom_op.general
  inputs : ['Unsqueeze_0_out0', 'MultiThreshold_0_param0']
  outputs: ['MultiThreshold_0_out0']

[4] Conv_0
  op_type: Conv
  domain : 
  inputs : ['Mul_0_out0', 'Conv_0_param0']
  outputs: ['Conv_0_out0']

[5] Pad_0
  op_type: Pad
  domain : 
  inputs : ['Mul_1_out0', 'Pad_0_param0', 'Pad_0_param1']
  outputs: ['Pad_0_out0']

[7] Conv_1
  op_type: Conv
  domain : 
  inputs : ['Pad_0_out0', 'Conv_1_param0']
  outputs: ['Conv_1_out0']

[11] MultiThreshold_1
  op_type: MultiThreshold
  domain : qonnx.custom_op.general